# Inital Setup
Load the models and configure tokenizer with chat template

In [1]:
import torch
from unsloth.chat_templates import get_chat_template
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configure BitsAndBytes for 4-bit fast inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "loknezmonzter/qwen-3.5-4b-medquad-lora",
    quantization_config=bnb_config,
    device_map="auto"
)

# Configure tokenizer and apply chat template
tokenizer = AutoTokenizer.from_pretrained("loknezmonzter/qwen-3.5-4b-medquad-lora")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


# Evaluation

In [4]:
import re
from datasets import load_dataset

# 1. THE EXTRACTION FUNCTION (Tweak this until it works here!)
def extract_medqa_answer(text):
    # 1. Strip the thinking block if it exists to avoid matching option lists
    clean_text = text.split("</think>")[-1] if "</think>" in text else text
    
    # 2. Look for the explicit final answer format
    match = re.search(r'(?:final answer|answer|choice|option)(?:\s*is)?[\s:]*(ANSWER:\s*[A-D])\b', clean_text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # 3. Fallback: If no explicit answer, find the last standalone A, B, C, or D 
    # but only in the last 150 characters (the conclusion)
    suffix = clean_text[-150:]
    matches = re.findall(r'\b([A-D])\b', suffix)
    if matches:
        return matches[-1].upper()
        
    return None


# 2. LOAD 2 SAMPLES
test_ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="test", streaming=True)
subset = test_ds.take(5)
samples = list(subset)

# 3. RUN TEST LOOP
val_map = {"A": "0", "B": "1", "C": "2", "D": "3", "0": "0", "1": "1", "2": "2", "3": "3"}

print(f"\033[1;36m=== MANUAL INSPECTION START ===\033[0m\n")

for i, item in enumerate(samples):
    # Format the prompt
    options_str = "\n".join([f"{k}) {v}" for k, v in item["options"].items()])
    prompt_content = f"{item['question']}\n{options_str}\n"

    print(f"\033[1mQuestion:\033[0m {prompt_content}\n")

    messages = [
        {
            "role": "system",
            "content": """
            You are a concise medical evaluator. 
            Complete your reasoning in 3 bullet points. Then, output the final answer.
            The final answer must always be in the below mentioned format: 
            
            ANSWER FORMAT
            --------------
            ANSWER: [LETTER]

            Here are some examples for the answer format:

            EXAMPLE 1
            ----------
            Question:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?
            A) Ampicillin
            B) Ceftriaxone
            C) Doxycycline
            D) Nitrofurantoin
            REASONING: The patient's symptoms of burning urination and a gravid uterus suggest a urinary tract infection during pregnancy. Nitrofurantoin is a safe and effective treatment for lower UTIs in pregnant women during the second trimester.
            ANSWER: D

            EXAMPLE 2
            ----------
            Question: A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had died only after she awoke in the morning. No cause of death was determined based on the autopsy. Which of the following precautions could have prevented the death of the baby?
            A) Placing the infant in a supine position on a firm mattress while sleeping
            B) Keeping the infant covered and maintaining a high room temperature
            C) Application of a device to maintain the sleeping position
            D) Avoiding pacifier use during sleep
            REASONING: The autopsy revealed no cause of death, indicating Sudden Infant Death Syndrome (SIDS). The most effective preventative measure against SIDS is placing the infant in a supine position on a firm mattress.
            ANSWER: A

            """
        },
        {
            "role": "user",
            "content": prompt_content
        }
    ]
    
    # Inference (Assumes model/tokenizer are already loaded in your notebook)
    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        inputs, 
        max_new_tokens=512,
        do_sample=True,
        repetition_penalty=1.0,
        top_p=0.5,
        pad_token_id=tokenizer.eos_token_id
    )
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()
    print(f"\033[1m\nRAW_OUTPUT:\033[0m {raw_output}\n")

    # Apply Debug Regex
    prediction = extract_medqa_answer(raw_output)
    ground_truth = str(item["answer_idx"])
    
    # Compare
    is_correct = val_map.get(prediction) == val_map.get(ground_truth)
    
    # OUTPUT
    print(f"\033[1m--- QUESTION {i+1} ---\033[0m")
    print(f"\033[1;34mGround Truth:\033[0m {ground_truth}")
    print(f"\033[1;34mRegex Extracted:\033[0m {prediction}")
    print(f"\033[1;34mVerdict:\033[0m {'✅ CORRECT' if is_correct else '❌ FAILED'}")
    print(f"\n\033[1;33mRaw Model Reasoning:\033[0m")
    print(f"{raw_output}")
    print("-" * 50 + "\n")

=== MANUAL INSPECTION START ===

Question: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
A) Disclose the error to the patient and put it in the operative report
B) Tell the attending that he cannot fail to disclose this mistake
C) Report the physician to the ethics committee
D) Refuse to dictate the operative report



RAW_OUTPUT: A) Disclose the error to the patient and put it in the operative report

--- QUESTION 1 ---
Ground Truth

# Cleanup
**NOTE**
Run this cell whenever you want to clear VRAM

In [ ]:
import gc
import torch

print("[CLEANUP] Unloading model from GPU...")

del model
del tokenizer

gc.collect() # Python Garbage Collector
torch.cuda.empty_cache() # PyTorch VRAM Release

print("VRAM cleared")

[CLEANUP] Unloading model from GPU...
VRAM cleared. Proceeding to next model.


In [1]:
!nvidia-smi

Wed May  6 10:48:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:06.0 Off |                    0 |
| 30%   32C    P8             25W /  300W |     349MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Manual Benchmarking

Generate questions from Gemini for testing the MedQuAD capabilities of the fine tuned model and then have Gemini evaluate the answers. 

In [2]:
from unsloth import FastVisionModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/qwen3.5-4B",
    max_seq_length=4096,
    load_in_4bit=False
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
FastVisionModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.1: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 44.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [3]:
questions = [
    """What is the underlying biological mechanism that causes the immune system to attack the thyroid gland in Hashimoto's thyroiditis?""",

    "What are the potential systemic complications of untreated chronic kidney disease (CKD) over a long-term period?",

    "How does the Varicella-Zoster virus reactivate to cause shingles, and what are the primary risk factors for this reactivation?",

    "What are the early clinical manifestations of Amyotrophic Lateral Sclerosis (ALS), and how do they differ from normal age-related muscle weakness?",

    "How is definitively diagnosing endometriosis achieved, and what are the limitations of standard imaging techniques like ultrasounds for this condition?",

    "What are the diagnostic criteria for preeclampsia during pregnancy, and what specific risks does it pose to maternal and fetal health?",

    "What are the accepted treatment options for managing refractory temporal lobe epilepsy in adult patients?",

    "What are the long-term side effects and risks associated with chronic corticosteroid use for severe autoimmune conditions?",

    "Are there significant contraindications or severe risks when a patient combines SSRI antidepressants with the over-the-counter supplement St. John's Wort?",
    
    "Is there a genetic inheritance pattern associated with Marfan syndrome, and what are the primary genetic mutations responsible?",

    """What is the standard prognosis for an individual diagnosed with Huntington's disease, and what are the distinct stages of its neurological progression?""",

    """What evidence-based lifestyle modifications and preventative measures can significantly reduce the risk of developing Type 2 Diabetes in prediabetic patients?""",

    """What are the current medical guidelines for colorectal cancer screening in average-risk individuals, and what exactly do the non-invasive stool-based tests detect?""",

    """How does celiac disease damage the small intestine, and what are the specific long-term nutritional deficiencies that can occur if a strict gluten-free diet is not maintained?""",

    """What is the expected recovery timeline, and what are the most common potential post-operative risks associated with coronary artery bypass grafting (CABG) surgery?""",
]

In [4]:
for i, q in enumerate(questions):
    print(f"\n--- Question {i+1} ---")
    print(f"Prompt: {q}")

    messages = [
        {"role": "user", "content": q}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        top_p=0.9,
        temperature=0.2
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"Response: {response}\n")
    print("-*" * 50)


--- Question 1 ---
Prompt: What is the underlying biological mechanism that causes the immune system to attack the thyroid gland in Hashimoto's thyroiditis?


Response: <think>:

</think>

The underlying biological mechanism driving Hashimoto's thyroiditis is **autoimmunity**, specifically a process where the immune system mistakenly identifies components of the thyroid gland as foreign invaders and launches a targeted attack. This process is complex and involves a cascade of events starting from genetic predisposition and culminating in tissue destruction.

### 1. Genetic Susceptibility and Environmental Triggers
The process rarely begins spontaneously without a trigger. Individuals with Hashimoto's often carry specific genetic markers, most notably the **HLA-DR5** allele, which influences how the immune system processes antigens. Environmental factors, such as exposure to iodine (either deficiency or excess), viral infections, or stress, can act as the "spark" that breaks tolerance. These triggers may cause the immune system to misidentify thyroid proteins as threats.

### 2. Loss of Tolerance and Antigen Presentation
Under normal conditio